# 05 Final Data Products and Executive Summary

This notebook prepares final analytical outputs for Tableau and presents a compact executive summary of national performance.

Deliverables from this notebook:
- Tableau-ready aggregate tables (year, state, crop, season).
- Headline KPI outputs for production, yield, growth, and volatility.
- Ranked lists to support strategy and policy discussion.

### Imports

Load `pandas` for final aggregation, KPI calculations, and export tasks.

In [2]:
import pandas as pd

### Load Cleaned Base Table

Read the cleaned dataset that will be used to build all final summary tables and KPI outputs.

In [3]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

### Add Derived Productivity Field

Create `production_per_area` as a direct per-row productivity metric for optional downstream use in dashboards.

In [4]:
df["production_per_area"] = df["production"] / df["area"]

### Build Annual Summary Table

Aggregate national totals by year and compute annual yield.

This table is the backbone for trend, growth, and latest-year KPI reporting.

In [10]:
yearly = df.groupby("year").agg({
    "production": "sum",
    "area": "sum"
}).reset_index()

yearly["yield"] = yearly["production"] / yearly["area"]

### Build State-Level Summary Table

Aggregate production and area by state-year and compute yield.

Used for state rankings and regional performance views.

In [6]:
state_level = df.groupby(["state", "year"]).agg({
    "production": "sum",
    "area": "sum"
}).reset_index()

state_level["yield"] = state_level["production"] / state_level["area"]

### Build Crop-Level Summary Table

Aggregate production and area by crop-year and compute yield.

Used for crop rankings, efficiency comparisons, and volatility analysis.

In [7]:
crop_level = df.groupby(["crop", "year"]).agg({
    "production": "sum",
    "area": "sum"
}).reset_index()

crop_level["yield"] = crop_level["production"] / crop_level["area"]

### Build Season-Level Summary Table

Aggregate production and area by season-year and compute yield.

This supports seasonal contribution analysis in Tableau.

In [8]:
season_level = df.groupby(["season", "year"]).agg({
    "production": "sum",
    "area": "sum"
}).reset_index()

season_level["yield"] = season_level["production"] / season_level["area"]

## Executive Summary

This section presents headline indicators designed for quick decision review.

Focus areas:
- National scale (`production`, `area`, `yield`)
- Current-period snapshot
- Top and bottom performers
- Growth trajectory and stability risk

In [11]:
total_production = df["production"].sum()
total_area = df["area"].sum()
overall_yield = total_production / total_area

print("=== GLOBAL KPIs ===")
print(f"Total Production: {total_production:,.0f}")
print(f"Total Area: {total_area:,.0f}")
print(f"Overall Yield: {overall_yield:.2f}")

=== GLOBAL KPIs ===
Total Production: 6,424,698,660
Total Area: 1,452,554,210
Overall Yield: 4.42


### Headline KPIs (Full Period)

Compute total production, total cultivated area, and overall yield for the full dataset window.

These metrics provide the top-line scale and efficiency baseline.

In [12]:
latest_year = yearly["year"].max()
latest_data = yearly[yearly["year"] == latest_year]

print(f"\n=== LATEST YEAR ({latest_year}) ===")
display(latest_data)


=== LATEST YEAR (2019) ===


,year,production,area,yield
7,2019,884629133.0,1.929420e+08,4.584948


### Latest-Year Snapshot

Show the most recent year from the annual summary table.

This gives a current-state reference point for all ranking and growth discussions.

In [13]:
top_states = (
    state_level.groupby("state")["production"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

print("\n=== TOP STATES BY PRODUCTION ===")
display(top_states)


=== TOP STATES BY PRODUCTION ===


,state,production
0,Uttar Pradesh,1.813314e+09
1,Maharashtra,8.185616e+08
2,Karnataka,4.645879e+08
3,Madhya Pradesh,4.552760e+08
4,Tamil Nadu,3.416085e+08
5,Gujarat,3.278981e+08
6,West Bengal,3.251977e+08
7,Punjab,3.049759e+08
8,Rajasthan,2.715197e+08
9,Andhra Pradesh,2.700791e+08


### Top Producing States

Rank states by cumulative production across the available years.

Used to identify high-contribution regions for targeted policy and investment attention.

In [14]:
top_crops = (
    crop_level.groupby("crop")["production"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

print("\n=== TOP CROPS BY PRODUCTION ===")
display(top_crops)


=== TOP CROPS BY PRODUCTION ===


,crop,production
0,Sugarcane,2.914037e+09
1,Rice,9.233879e+08
2,Wheat,8.627091e+08
3,Potato,2.836033e+08
4,Cotton(lint),2.310621e+08
5,Maize,2.179480e+08
6,Banana,1.119635e+08
7,Soyabean,8.772331e+07
8,Jute,8.051538e+07
9,Bajra,7.868335e+07


### Top Producing Crops

Rank crops by cumulative production to identify nationally dominant crops.

This helps prioritize high-impact crop categories for policy planning.

In [15]:
efficient_crops = (
    crop_level.groupby("crop")["yield"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

print("\n=== MOST EFFICIENT CROPS (YIELD) ===")
display(efficient_crops)


=== MOST EFFICIENT CROPS (YIELD) ===


,crop,yield
0,Sugarcane,75.223573
1,Banana,37.215509
2,Tapioca,32.171481
3,Potato,22.029788
4,Onion,15.741971
5,Jute,14.090035
6,Ginger,12.345275
7,Sweet potato,9.825643
8,Mesta,8.200682
9,other oilseeds,7.052935


### Most Efficient Crops (High Yield)

Rank crops by average yield to surface strong efficiency performers.

These crops can inform replication of successful agronomic practices.

In [16]:
inefficient_crops = (
    crop_level.groupby("crop")["yield"]
    .mean()
    .sort_values(ascending=True)
    .head(10)
    .reset_index()
)

print("\n=== LEAST EFFICIENT CROPS ===")
display(inefficient_crops)


=== LEAST EFFICIENT CROPS ===


,crop,yield
0,Cardamom,0.270706
1,Moth,0.314256
2,Niger seed,0.418661
3,Other Summer Pulses,0.440220
4,Sesamum,0.451606
5,Guar seed,0.495440
6,Moong(Green Gram),0.499948
7,Horse-gram,0.505490
8,Linseed,0.552493
9,Black pepper,0.564252


### Least Efficient Crops (Low Yield)

Rank crops with the lowest average yield.

These indicate potential intervention zones for productivity improvement.

In [17]:
start_year = yearly["year"].min()
end_year = yearly["year"].max()

start_val = yearly[yearly["year"] == start_year]["production"].values[0]
end_val = yearly[yearly["year"] == end_year]["production"].values[0]

years = end_year - start_year

cagr = ((end_val / start_val) ** (1 / years) - 1) * 100

print("\n=== GROWTH METRICS ===")
print(f"Period: {start_year} → {end_year}")
print(f"CAGR (Production): {cagr:.2f}%")


=== GROWTH METRICS ===
Period: 2011 → 2019
CAGR (Production): 2.10%


### Growth Indicator (CAGR)

Compute compound annual growth rate of production from first to last year.

CAGR summarizes the long-run growth trajectory in a single metric.

In [18]:
yield_volatility = (
    crop_level.groupby("crop")["yield"]
    .std()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

print("\n=== MOST VOLATILE CROPS (YIELD STD DEV) ===")
display(yield_volatility)


=== MOST VOLATILE CROPS (YIELD STD DEV) ===


,crop,yield
0,Banana,6.972624
1,other oilseeds,5.614208
2,Sugarcane,4.127311
3,Ginger,3.678067
4,Mesta,2.575982
5,Tapioca,2.327119
6,Potato,2.306381
7,Onion,2.156128
8,Garlic,1.583398
9,Oilseeds total,1.037991


### Export Final Tableau Inputs and Closing

Save all summary tables for Tableau model inputs:
- yearly summary
- state-level summary
- crop-level summary
- season-level summary

These files, together with the executive KPI outputs above, complete the final reporting package.

### Volatility Risk View

Identify crops with the highest yield standard deviation.

High volatility can indicate higher production risk and may require resilience-focused interventions.

In [20]:
yearly.to_csv("../data/processed/yearly.csv", index=False)
state_level.to_csv("../data/processed/state_level.csv", index=False)
crop_level.to_csv("../data/processed/crop_level.csv", index=False)
season_level.to_csv("../data/processed/season_level.csv", index=False)

### Executive Closing Note

The KPI outputs above summarize scale, efficiency, growth, and volatility in one review flow.

Use these tables directly in Tableau to build the final stakeholder-facing dashboard and report visuals.

### Export Final Tableau Inputs

Save all prepared summary tables as CSV files for dashboard integration:
- yearly
- state-level
- crop-level
- season-level